# 구조화된 데이터 파싱하기 - Pydantic과 JSON으로 변환해요

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- `PydanticOutputParser`로 LLM 출력을 타입이 보장된 Pydantic 객체로 변환할 수 있어요
- `JsonOutputParser`로 LLM 출력을 Python 딕셔너리(dict)로 변환할 수 있어요
- 두 파서의 차이를 이해하고 상황에 맞게 선택할 수 있어요
- `with_structured_output()` 메서드로 더 간편하게 구조화된 출력을 받을 수 있어요

## 사전 지식

- **Pydantic**: Python 데이터 검증 라이브러리예요. `BaseModel`을 상속해 클래스를 정의하면 필드 타입과 제약조건을 자동으로 검사해줘요
- **JSON(JavaScript Object Notation)**: 키-값 쌍으로 이루어진 경량 데이터 형식이에요. Python의 `dict`와 1:1로 대응돼요

## 파이프라인 흐름

```mermaid
flowchart LR
    A["PromptTemplate<br/>(형식 지침 포함)"] -->|"프롬프트 생성"| B["ChatOpenAI<br/>(LLM)"]
    B -->|"JSON 텍스트 반환"| C{"파서 선택"}
    C -->|"PydanticOutputParser"| D["Pydantic 객체<br/>result.title, result.author"]
    C -->|"JsonOutputParser"| E["Python dict<br/>result.title, result.author"]
    C -->|"with_structured_output()"| F["Pydantic 객체<br/>(Function Calling 활용)"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class A input
    class B,C process
    class D,E,F output
```

> **핵심 포인트**: `get_format_instructions()`가 LLM에게 "이런 JSON 형식으로 답해줘"라고 지시해요. 파서는 그 응답을 받아 Python 객체로 변환해줘요.

## 다루는 파서

| 파서 | 출력 타입 | 타입 검증 | 권장 상황 |
|------|-----------|-----------|-----------|
| **PydanticOutputParser** | Pydantic 객체 | 강력 | 프로덕션, 복잡한 검증 |
| **JsonOutputParser** | Python dict | 기본 | 프로토타이핑, 단순 구조 |

In [1]:
from dotenv import load_dotenv

load_dotenv()


True

# 1. PydanticOutputParser

`PydanticOutputParser`는 Pydantic 모델을 사용해 LLM 출력을 **타입 검증이 보장된 객체**로 변환해요.

## 주요 메서드

- `get_format_instructions()`: LLM에게 줄 JSON 형식 지침을 생성해요. 이걸 프롬프트에 포함시켜야 LLM이 올바른 형식으로 응답해요
- `parse()`: LLM이 반환한 문자열을 Pydantic 객체로 변환해요

## 실무 팁

`Field(description="...")` 설명이 구체적일수록 LLM이 더 정확한 값을 채워줘요. 예를 들어 `"평점 (1-5점)"` 대신 `"1부터 5 사이의 정수 평점"`처럼 범위와 타입을 함께 명시해요.

## 실용 예제: 도서 정보 추출

온라인 서점에서 도서 리뷰를 분석하여 핵심 정보를 추출하는 시나리오입니다.


In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# 1단계: Pydantic 모델 정의
class BookReview(BaseModel):
    """도서 리뷰에서 추출한 정보"""
    
    title: str = Field(description="도서 제목")
    author: str = Field(description="저자명")
    rating: int = Field(description="평점 (1-5점)", ge=1, le=5)
    genres: List[str] = Field(description="장르 목록 (최대 3개)")
    summary: str = Field(description="한 문장 요약")

# 2단계: 모델 및 파서 초기화
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# TODO: PydanticOutputParser를 생성하세요 (pydantic_object=BookReview)

parser = PydanticOutputParser(pydantic_object=BookReview)

print("=" * 60)
print("📋 파서가 생성한 형식 지침")
print("=" * 60)
# TODO: parser.get_format_instructions()를 출력하세요
parser.get_format_instructions()


📋 파서가 생성한 형식 지침


'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"description": "도서 리뷰에서 추출한 정보", "properties": {"title": {"description": "도서 제목", "title": "Title", "type": "string"}, "author": {"description": "저자명", "title": "Author", "type": "string"}, "rating": {"description": "평점 (1-5점)", "maximum": 5, "minimum": 1, "title": "Rating", "type": "integer"}, "genres": {"description": "장르 목록 (최대 3개)", "items": {"type": "string"}, "title": "Genres", "type": "array"}, "summary": {"description": "한 문장 요약", "title": "Summary", "type": "string"}}, "required": ["title", "author", "rating", "genres", "summ

In [5]:
from langchain_core.output_parsers import format_instructions
# ---------------------------------------------------
# 프롬프트 템플릿에 형식 지침을 주입하고 체인 구성
# ---------------------------------------------------

# 1단계: 프롬프트 템플릿 작성
# {format_instructions}: LLM에게 JSON 형식을 알려주는 자리
# TODO: PromptTemplate.from_template()으로 프롬프트를 작성하세요
prompt = PromptTemplate.from_template(
    """
    다음 텍스트에서 도서 정보를 추출하시오.

    텍스트:
    {text}

    {format_instructions}
    """
)

# 2단계: partial()로 형식 지침을 미리 채워 넣기
# partial(): 일부 변수만 미리 고정해두는 메서드 (나머지 변수는 invoke 시 전달)
# TODO: prompt.partial()로 format_instructions를 주입하세요
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# 3단계: 체인 구성 (프롬프트 | LLM | Pydantic 파서)
# TODO: LCEL 파이프라인으로 체인을 구성하세요
chain = (
  prompt
  | model
  | parser
)


In [6]:
# ---------------------------------------------------
# 체인 실행 - 도서 리뷰 텍스트에서 구조화된 정보 추출
# ---------------------------------------------------

# 도서 리뷰 텍스트 (자연어 입력)
review_text = """
'파이썬 마스터하기'는 김파이썬 저자의 역작입니다. 
초보자부터 고급 개발자까지 모두에게 유용한 내용을 담고 있으며, 
실용적인 예제가 풍부합니다. 특히 데이터 분석과 웹 개발 파트가 인상적이었습니다.
5점 만점에 5점을 주고 싶습니다!
"""

# invoke(): 체인 실행 → Pydantic 객체로 반환
# TODO: chain.invoke()로 실행하세요
res = chain.invoke({"text": review_text})

print("\n" + "=" * 60)
print("📖 추출된 도서 정보")
print("=" * 60)
# Pydantic 객체: 점 표기법으로 필드에 접근 (result.title, result.author 등)
# TODO: result의 각 필드를 출력하세요
res



📖 추출된 도서 정보


BookReview(title='파이썬 마스터하기', author='김파이썬', rating=5, genres=['프로그래밍', '데이터 분석', '웹 개발'], summary='초보자부터 고급 개발자까지 모두에게 유용한 내용을 담고 있는 실용적인 예제가 풍부한 도서입니다.')

In [7]:
res.title
print(f' ==> [Line 1]: \033[38;2;93;62;46m[res.title]\033[0m({type(res.title).__name__}) = \033[38;2;115;124;87m{res.title}\033[0m')
res.author
print(f' ==> [Line 3]: \033[38;2;65;21;17m[res.author]\033[0m({type(res.author).__name__}) = \033[38;2;74;233;168m{res.author}\033[0m')
res.rating
print(f' ==> [Line 5]: \033[38;2;185;77;56m[res.rating]\033[0m({type(res.rating).__name__}) = \033[38;2;18;255;72m{res.rating}\033[0m')
res.genres
print(f' ==> [Line 7]: \033[38;2;100;159;223m[res.genres]\033[0m({type(res.genres).__name__}) = \033[38;2;3;30;154m{res.genres}\033[0m')
res.summary
print(f' ==> [Line 9]: \033[38;2;15;103;61m[res.summary]\033[0m({type(res.summary).__name__}) = \033[38;2;151;191;195m{res.summary}\033[0m')


 ==> [Line 1]: [res.title](str) = 파이썬 마스터하기
 ==> [Line 3]: [res.author](str) = 김파이썬
 ==> [Line 5]: [res.rating](int) = 5
 ==> [Line 7]: [res.genres](list) = ['프로그래밍', '데이터 분석', '웹 개발']
 ==> [Line 9]: [res.summary](str) = 초보자부터 고급 개발자까지 모두에게 유용한 내용을 담고 있는 실용적인 예제가 풍부한 도서입니다.


## with_structured_output(): 더 간편한 방법이에요

LangChain 최신 버전에서는 `with_structured_output()` 메서드로 파서 없이 구조화된 출력을 받을 수 있어요. 내부적으로 Function Calling(함수 호출, LLM이 정해진 스키마로만 응답하도록 강제하는 기능)을 활용해서 더 안정적이에요.

> **주의**: `with_structured_output()`은 Function Calling을 지원하는 모델(GPT-4, GPT-4o 등)에서만 동작해요. 지원하지 않는 모델에서는 `PydanticOutputParser`를 사용해요.

In [9]:
# with_structured_output()을 사용한 간편한 방법
# TODO: ChatOpenAI 모델을 생성하고 .with_structured_output(BookReview)를 호출하세요
model_with_structure = ChatOpenAI(
  model="gpt-4o-mini",
  temperature=0
).with_structured_output(BookReview)

# 직접 호출 - 프롬프트 없이도 구조화된 출력 가능
# TODO: model_with_structure.invoke()로 실행하세요
res = model_with_structure.invoke(
  f"""
  다음 텍스트에서 도서 정보를 추출하시오 
  {review_text}
  """
)

print("=" * 60)
print("✨ with_structured_output() 결과")
print("=" * 60)
# TODO: 결과의 제목, 저자, 평점을 출력하세요
res


✨ with_structured_output() 결과


BookReview(title='파이썬 마스터하기', author='김파이썬', rating=5, genres=['프로그래밍', '데이터 분석', '웹 개발'], summary='초보자부터 고급 개발자까지 모두에게 유용한 내용을 담고 있으며, 실용적인 예제가 풍부합니다. 특히 데이터 분석과 웹 개발 파트가 인상적이었습니다.')

# 2. JsonOutputParser

`JsonOutputParser`는 LLM 출력을 Python `dict`로 변환해요. Pydantic 스키마를 선택적으로 사용하거나 아예 없이도 쓸 수 있어 프로토타이핑에 편리해요.

## Pydantic 스키마와 함께 사용하기 vs. 스키마 없이 사용하기

| 방법 | 장점 | 단점 |
|------|------|------|
| 스키마 포함 | 형식 지침 자동 생성, 검증 강화 | Pydantic 모델 정의 필요 |
| 스키마 없음 | 코드 간결, 유연한 구조 | 프롬프트에서 구조를 직접 설명해야 함 |

> **실무 팁**: 빠른 프로토타이핑에는 스키마 없이 시작하고, 코드가 안정화되면 Pydantic 스키마로 전환하는 방법이 효과적이에요.

## 실용 예제 1: Pydantic 스키마와 함께 사용

운동 루틴을 생성하고 JSON으로 파싱하는 시나리오입니다.


In [14]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

# 1단계: Pydantic 모델 정의
class WorkoutRoutine(BaseModel):
    """운동 루틴 정보"""
    
    routine_name: str = Field(description="운동 루틴 이름")
    target_muscle: str = Field(description="타겟 근육군")
    exercises: List[str] = Field(description="운동 목록 (3-5개)")
    duration_minutes: int = Field(description="총 소요 시간 (분)")

# 2단계: JsonOutputParser 생성 (Pydantic 스키마 포함)
# TODO: JsonOutputParser를 생성하세요 (pydantic_object=WorkoutRoutine)
json_parser = JsonOutputParser(pydantic_object=WorkoutRoutine)


# 3단계: 프롬프트 템플릿
# TODO: ChatPromptTemplate.from_messages()로 프롬프트를 구성하세요
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 운동 전문가 입니다. 사용자의 요청에 맞는 운동 루틴을 작성하시오"),
    ("human", "{request}\n\n{format_instructions}")
])


# TODO: partial()로 형식 지침을 주입하세요
json_prompt = json_prompt.partial(format_instructions=json_parser.get_format_instructions())

# 4단계: 체인 구성
# TODO: LCEL 파이프라인으로 체인을 구성하세요
chain = (
    json_prompt
    | model
    | json_parser
)

# 5단계: 실행
# TODO: chain.invoke()로 실행하세요
res = chain.invoke({"request": "한 달 안에 몸짱이 되고 싶어"})

print("=" * 60)
print("💪 생성된 운동 루틴 (JSON)")
print("=" * 60)
# TODO: 결과를 dict 키 표기법으로 출력하세요 (workout['routine_name'] 등)
res

💪 생성된 운동 루틴 (JSON)


{'routine_name': '한 달 몸짱 만들기',
 'target_muscle': '전신',
 'exercises': ['스쿼트 3세트 x 12회',
  '푸시업 3세트 x 10회',
  '덤벨 로우 3세트 x 12회',
  '플랭크 3세트 x 30초',
  '버피 3세트 x 10회'],
 'duration_minutes': 45}

## 실용 예제 2: 스키마 없이 사용하기

Pydantic 모델 없이도 `JsonOutputParser`를 사용할 수 있습니다. 이 경우 프롬프트에서 원하는 JSON 구조를 명시해야 합니다.


In [17]:
# ---------------------------------------------------
# 스키마 없이 JsonOutputParser 사용 - 프롬프트에서 구조 명시
# ---------------------------------------------------

# JsonOutputParser(): Pydantic 없이도 사용 가능 (유연한 프로토타이핑용)
# TODO: JsonOutputParser를 인자 없이 생성하세요
simple_json_parser = JsonOutputParser()

# 프롬프트에서 직접 JSON 구조 명시
# 스키마가 없으므로 원하는 필드를 프롬프트 안에 직접 설명해야 함
# TODO: ChatPromptTemplate.from_messages()로 프롬프트를 구성하세요
#       응답 형식에 recipe_name, ingredients, cooking_time, difficulty를 명시하세요
simple_json_prompt = ChatPromptTemplate.from_messages([
  ("system", "당신은 운동 전문가 입니다. 사용자의 요청에 맞는 운동 루틴을 작성하시오"),
  ("human", "{request}\n\n{format_instructions}")
])

# partial()로 형식 지침 주입
# TODO: partial()로 format_instructions를 주입하세요
simple_json_prompt = simple_json_prompt.partial(format_instructions=simple_json_parser.get_format_instructions())

# 체인 구성
# TODO: LCEL 파이프라인으로 체인을 구성하세요
chain = (
  simple_json_prompt
  | model
  | simple_json_parser
)

# 실행
# TODO: chain.invoke()로 실행하세요
res = chain.invoke({"request": "한 달 안에 몸짱이 되고 싶어"})

print("\n" + "=" * 60)
print("🍝 레시피 정보 (스키마 없는 JSON)")
print("=" * 60)
# dict 타입: 키 표기법으로 접근 (result['key'])
# TODO: 결과를 출력하세요
res



🍝 레시피 정보 (스키마 없는 JSON)


{'goal': '몸짱',
 'duration': '1개월',
 'workout_routine': {'weekly_schedule': [{'day': '월요일',
    'workout': [{'exercise': '스쿼트', 'sets': 4, 'reps': 12},
     {'exercise': '벤치프레스', 'sets': 4, 'reps': 10},
     {'exercise': '풀업', 'sets': 3, 'reps': 8},
     {'exercise': '플랭크', 'sets': 3, 'duration': '30초'}]},
   {'day': '화요일',
    'workout': [{'exercise': '유산소 운동 (조깅 또는 자전거)', 'duration': '30분'},
     {'exercise': '복근 운동 (크런치, 레그 레이즈)', 'sets': 3, 'reps': 15}]},
   {'day': '수요일',
    'workout': [{'exercise': '데드리프트', 'sets': 4, 'reps': 10},
     {'exercise': '덤벨 숄더 프레스', 'sets': 4, 'reps': 10},
     {'exercise': '바벨 로우', 'sets': 3, 'reps': 10}]},
   {'day': '목요일',
    'workout': [{'exercise': '유산소 운동 (HIIT)', 'duration': '20분'},
     {'exercise': '스트레칭', 'duration': '15분'}]},
   {'day': '금요일',
    'workout': [{'exercise': '런지', 'sets': 4, 'reps': 12},
     {'exercise': '푸쉬업', 'sets': 4, 'reps': 10},
     {'exercise': '트라이셉스 딥', 'sets': 3, 'reps': 10}]},
   {'day': '토요일',
    'workout': [{'

## 핵심 요약

### PydanticOutputParser vs JsonOutputParser

| 항목 | PydanticOutputParser | JsonOutputParser |
|------|---------------------|------------------|
| 출력 타입 | Pydantic 객체 | Python dict |
| 타입 검증 | 강력 (Pydantic 기반) | 기본 (JSON 스키마) |
| 속성 접근 | `result.title` (점 표기법) | `result['title']` (키 표기법) |
| 권장 상황 | 프로덕션, 복잡한 검증 | 프로토타이핑, 단순 구조 |

### 파서 선택 의사결정

```mermaid
flowchart TD
    Q1{"타입 안전성과<br/>자동 검증이 필요한가요?"}
    Q2{"빠른 프로토타이핑이<br/>목적인가요?"}
    Q3{"Function Calling을<br/>지원하는 모델인가요?"}
    A1["PydanticOutputParser"]
    A2["JsonOutputParser<br/>(스키마 없음)"]
    A3["with_structured_output()"]
    A4["PydanticOutputParser<br/>또는 JsonOutputParser"]

    Q1 -->|예| Q3
    Q1 -->|아니오| Q2
    Q2 -->|예| A2
    Q2 -->|아니오| A4
    Q3 -->|예| A3
    Q3 -->|아니오| A1

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class Q1,Q2,Q3 process
    class A1,A2,A3,A4 output
```

## 다음 노트북 예고

**노트북 03 - CommaSeparatedListOutputParser** 에서는 복잡한 스키마 정의 없이 쉼표로 구분된 리스트를 바로 얻는 방법을 배워요. 키워드 추출, 추천 목록 생성 등 간단한 배열 데이터에 최적이에요.

## 연습 문제

다음 연습 문제를 통해 `PydanticOutputParser`와 `JsonOutputParser`의 활용법을 직접 실습해봅시다.

### 연습 1: 영화 리뷰 분석기 (PydanticOutputParser)

`PydanticOutputParser`를 사용하여 영화 리뷰 텍스트에서 구조화된 정보를 추출하는 체인을 만들어보세요.

**요구사항:**
- Pydantic 모델 `MovieReview`를 정의하세요:
  - `title` (str): 영화 제목
  - `reviewer_sentiment` (str): 리뷰어의 감정 (긍정/부정/중립)
  - `score` (int): 평점 (1~10), `ge=1`, `le=10` 제약조건 사용
  - `pros` (List[str]): 장점 목록
  - `cons` (List[str]): 단점 목록
- 아래 리뷰 텍스트를 분석하세요:

```
"인셉션은 크리스토퍼 놀란 감독의 걸작입니다. 꿈 속의 꿈이라는 독창적인 설정이 인상적이고,
시각 효과가 압도적입니다. 한스 짐머의 음악도 영화의 긴장감을 극대화합니다.
다만 스토리가 다소 복잡하여 한 번에 이해하기 어려울 수 있고,
감정선이 약하다는 의견도 있습니다. 10점 만점에 9점을 주고 싶습니다."
```

In [ ]:
# ============================================================
# TODO: 영화 리뷰 분석기 만들기 (PydanticOutputParser)
# 힌트: MovieReview Pydantic 모델 정의 → PydanticOutputParser 생성 →
#       prompt.partial()로 형식 지침 주입 → 체인 구성 → invoke()
# 예상 결과: MovieReview 타입 객체로 제목/감정/평점/장단점이 추출됨
# ============================================================
class MovieReview(BaseModel):
    """영화 리뷰 분석기"""
    title: str = Field(description="영화 제목")
    reviewer_sentiment: str = Field(description="리뷰어의 감정 (긍정/부정/중립)")
    score: int = Field(description="평점 (1~10)", ge=1, le=10)
    pros: List[str] = Field(description="장점 목록")
    cons: List[str] = Field(description="단점 목록")


model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

parser = PydanticOutputParser(pydantic_object=MovieReview)

prompt = PromptTemplate.from_template(
    """
    {review}\n\n{format_instructions}
    """
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = (
    prompt
    | model
    | parser
)

review_txt = """
인셉션은 크리스토퍼 놀란 감독의 걸작입니다. 꿈 속의 꿈이라는 독창적인 설정이 인상적이고,
시각 효과가 압도적입니다. 한스 짐머의 음악도 영화의 긴장감을 극대화합니다.
다만 스토리가 다소 복잡하여 한 번에 이해하기 어려울 수 있고,
감정선이 약하다는 의견도 있습니다. 10점 만점에 9점을 주고 싶습니다.
"""

res = chain.invoke({"review": review_txt})

res

MovieReview(title='인셉션', reviewer_sentiment='긍정', score=9, pros=['독창적인 설정인 꿈 속의 꿈', '압도적인 시각 효과', '한스 짐머의 긴장감 넘치는 음악'], cons=['복잡한 스토리로 한 번에 이해하기 어려움', '약한 감정선'])

### 연습 2: 맛집 정보 추출기 (JsonOutputParser)

`JsonOutputParser`를 **스키마 없이** 사용하여 맛집 리뷰에서 정보를 JSON으로 추출하는 체인을 만들어보세요.

**요구사항:**
- `JsonOutputParser()`를 스키마 없이 생성하세요
- 프롬프트에서 원하는 JSON 구조를 직접 명시하세요:
  - `restaurant_name`: 식당 이름
  - `cuisine_type`: 음식 종류
  - `price_range`: 가격대 (저렴/보통/고급)
  - `recommended_dishes`: 추천 메뉴 목록 (배열)
  - `overall_rating`: 총평 (한 문장)
- 테스트 텍스트: "강남에 있는 '미소라멘'은 일본 현지 맛을 그대로 재현한 라멘 전문점입니다. 돈코츠라멘과 차슈덮밥이 특히 인기가 많으며, 가격은 1만원 내외로 합리적입니다. 점심시간에는 항상 줄을 서야 하지만 그만한 가치가 있습니다."

In [22]:
# ============================================================
# TODO: 맛집 정보 추출기 만들기 (JsonOutputParser - 스키마 없이)
# 힌트: JsonOutputParser()를 인자 없이 생성하고,
#       프롬프트에 원하는 JSON 필드를 직접 설명하세요
# 예상 결과: dict 타입으로 식당명/음식종류/가격대/추천메뉴/총평이 추출됨
# ============================================================
parser = JsonOutputParser()
prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 리뷰로부터 정보를 JSON 으로 추출하시오"),
    ("human", """
    리뷰:
    {review}

    출력형식:
    - `restaurant_name`: 식당 이름
    - `cuisine_type`: 음식 종류
    - `price_range`: 가격대 (저렴/보통/고급)
    - `recommended_dishes`: 추천 메뉴 목록 (배열)
    - `overall_rating`: 총평 (한 문장)
    """)
])

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = (
    prompt
    | model
    | parser
)

test_review = """
강남에 있는 '미소라멘'은 일본 현지 맛을 그대로 재현한 라멘 전문점입니다. 돈코츠라멘과 차슈덮밥이 특히 인기가 많으며, 가격은 1만원 내외로 합리적입니다. 점심시간에는 항상 줄을 서야 하지만 그만한 가치가 있습니다.
"""

res = chain.invoke({"review": test_review})
res


{'restaurant_name': '미소라멘',
 'cuisine_type': '라멘',
 'price_range': '보통',
 'recommended_dishes': ['돈코츠라멘', '차슈덮밥'],
 'overall_rating': '일본 현지 맛을 그대로 재현한 라멘 전문점으로, 점심시간에는 항상 줄을 서야 하지만 그만한 가치가 있습니다.'}

### 연습 3: with_structured_output()으로 이력서 정보 추출

`with_structured_output()` 메서드를 사용하여 이력서 텍스트에서 구조화된 정보를 추출해보세요.

**요구사항:**
- Pydantic 모델 `ResumeInfo`를 정의하세요:
  - `name` (str): 이름
  - `years_of_experience` (int): 경력 년수
  - `skills` (List[str]): 보유 기술 목록
  - `education` (str): 최종 학력
- `with_structured_output()`을 사용하여 OutputParser 없이 구조화된 출력을 받으세요
- 테스트 텍스트: "김개발은 서울대학교 컴퓨터공학과를 졸업하고 7년간 백엔드 개발자로 근무했습니다. Python, Java, Docker, Kubernetes에 능숙하며 최근에는 AI/ML 프로젝트에도 참여하고 있습니다."

In [24]:
# ============================================================
# TODO: with_structured_output()으로 이력서 정보 추출하기
# 힌트: model.with_structured_output(ResumeInfo)로 모델을 설정하고
#       OutputParser 없이 invoke()를 직접 호출하세요
# 예상 결과: ResumeInfo 타입 객체로 이름/경력/기술/학력이 추출됨
# ============================================================

class ResumeInfo(BaseModel):
    """이력서 정보 추출"""
    name: str = Field(description="이름")
    years_of_experience: int = Field(description="경력 년수")
    skills: List[str] = Field(description="보유 기술 목록")
    education: str = Field(description="최종 학력")

model_with_structured = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
).with_structured_output(ResumeInfo)


test_str = """
김개발은 서울대학교 컴퓨터공학과를 졸업하고 7년간 백엔드 개발자로 근무했습니다. Python, Java, Docker, Kubernetes에 능숙하며 최근에는 AI/ML 프로젝트에도 참여하고 있습니다.
"""

res = model_with_structured.invoke(
    f"""
    다음 이력서에서 정보를 추출하시오 
    {test_str}
    """
)

res

ResumeInfo(name='김개발', years_of_experience=7, skills=['Python', 'Java', 'Docker', 'Kubernetes', 'AI/ML'], education='서울대학교 컴퓨터공학과')